# Path Manipulation

Everything in this chapter works on the path alone and never calls AWS, so it runs without credentials: build a path, read its parts, join, change, make relative paths, compare, sort and serialize.

## Setup

No AWS is needed. `show_error` prints an exception in one line, so the demos can show what goes wrong.

In [1]:
from s3pathlib.api import S3Path
from s3pathlib.api import exc


def show_error(func):
    try:
        func()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

## 1. Build a path

`S3Path(bucket, *parts)`. The first argument is the bucket; it can also be an S3 URI or an S3 ARN, which carry the key with them. The other arguments are parts of the key. Repeated `/` are collapsed and empty parts are ignored.

In [2]:
print(S3Path("my-bucket", "folder", "file.txt"))
print(S3Path("my-bucket/folder/file.txt"))
print(S3Path("s3://my-bucket/folder/file.txt"))
print(S3Path("arn:aws:s3:::my-bucket/folder/file.txt"))
print(S3Path("my-bucket", "folder//sub/", "", "file.txt"))

S3Path('s3://my-bucket/folder/file.txt')
S3Path('s3://my-bucket/folder/file.txt')
S3Path('s3://my-bucket/folder/file.txt')
S3Path('s3://my-bucket/folder/file.txt')
S3Path('s3://my-bucket/folder/sub/file.txt')


The **last non-empty part decides the type**: a trailing `/` makes a folder (a "dir"), otherwise it is a file.

In [3]:
print(S3Path("my-bucket", "folder", "data"))
print(S3Path("my-bucket", "folder", "data/"))
print(S3Path("my-bucket", "folder/", ""))

S3Path('s3://my-bucket/folder/data')
S3Path('s3://my-bucket/folder/data/')
S3Path('s3://my-bucket/folder/')


`from_s3_uri` and `from_s3_arn` accept only that format, which is useful when the input comes from users. The constructor checks the format of a URI or ARN, not the bucket naming rules (S3 is the authority on names, and old buckets may break today's rules); use `validate_s3_bucket` to check a name yourself.

In [4]:
print(S3Path.from_s3_uri("s3://my-bucket/folder/"))
print(S3Path.from_s3_arn("arn:aws-cn:s3:::my-bucket/folder/file.txt"))
show_error(lambda: S3Path.from_s3_uri("my-bucket/folder/"))
show_error(lambda: S3Path("https://my-bucket.s3.amazonaws.com/file.txt"))
show_error(lambda: S3Path(1))

S3Path('s3://my-bucket/folder/')
S3Path('s3://my-bucket/folder/file.txt')
S3ValidationError: S3 URI must start with 's3://', got 'my-bucket/folder/'
S3ValidationError: bucket 'https:' in 'https://my-bucket.s3.amazonaws.com/file.txt' contains ':', which is not allowed in any bucket name
TypeError: the first argument has to be a str or an S3Path, got int: 1


In [5]:
from s3pathlib.api import validate_s3_bucket

show_error(lambda: validate_s3_bucket("My_Bucket"))

S3ValidationError: Bucket names can consist only of lowercase letters, numbers, dots (.), and hyphens (-). invalid char found ['B', 'M', '_']


## 2. The six kinds of path

Every `S3Path` is one of six kinds. The three with a bucket are absolute: a file (an S3 object), a dir (a logical folder) and the bucket itself (the root dir). The three without a bucket are relative: a relative file, a relative dir, and the void path, the empty relative path. `S3Path.make_relpath` builds relative paths, and `S3Path()` is void.

In [6]:
paths = [
    S3Path("my-bucket", "folder", "file.txt"),
    S3Path("my-bucket", "folder/"),
    S3Path("my-bucket"),
    S3Path.make_relpath("folder", "file.txt"),
    S3Path.make_relpath("folder/"),
    S3Path(),
]
print(f"{'path':<40} file   dir    bucket relpath void")
for p in paths:
    print(
        f"{p!r:<40} {p.is_file()!s:<6} {p.is_dir()!s:<6} {p.is_bucket()!s:<6} "
        f"{p.is_relpath()!s:<7} {p.is_void()!s:<5}"
    )

path                                     file   dir    bucket relpath void
S3Path('s3://my-bucket/folder/file.txt') True   False  False  False   False
S3Path('s3://my-bucket/folder/')         False  True   False  False   False
S3Path('s3://my-bucket/')                False  True   True   False   False
S3RelPath('folder/file.txt')             True   False  False  True    False
S3RelPath('folder/')                     False  True   False  True    False
S3VoidPath()                             False  False  False  True    True 


A relative path has a key but no bucket, so its `uri` and `arn` are `None`. The key of a bucket and of void is `""`, the same as `Prefix=""` in the S3 API.

In [7]:
print(f"{'path':<40} {'bucket':<10} {'key':<18} uri")
for p in paths:
    print(f"{p!r:<40} {p.bucket!r:<10} {p.key!r:<18} {p.uri!r}")

path                                     bucket     key                uri
S3Path('s3://my-bucket/folder/file.txt') 'my-bucket' 'folder/file.txt'  's3://my-bucket/folder/file.txt'
S3Path('s3://my-bucket/folder/')         'my-bucket' 'folder/'          's3://my-bucket/folder/'
S3Path('s3://my-bucket/')                'my-bucket' ''                 's3://my-bucket/'
S3RelPath('folder/file.txt')             None       'folder/file.txt'  None
S3RelPath('folder/')                     None       'folder/'          None
S3VoidPath()                             None       ''                 None


The `ensure_*` methods check the kind and raise an exception of `s3pathlib.exc`, handy at the top of your own functions. Each one also inherits a builtin exception (`S3PathIsNotFolderError` is a `TypeError`), so a generic `except` still works.

In [8]:
p = S3Path("my-bucket", "folder", "file.txt")
p.ensure_file()
p.ensure_not_relpath()
show_error(p.ensure_dir)
show_error(p.ensure_bucket)
show_error(p.ensure_relpath)
show_error(S3Path("my-bucket", "folder/").ensure_not_dir)
isinstance(exc.S3PathIsNotFolderError(), TypeError)

S3PathIsNotFolderError: 's3://my-bucket/folder/file.txt' is not a dir!
S3PathIsNotBucketError: 's3://my-bucket/folder/file.txt' is not a bucket!
S3PathIsNotRelpathError: 's3://my-bucket/folder/file.txt' is not a relpath!
S3PathTypeError: 's3://my-bucket/folder/' is a dir!


True

A check that passes returns `None`. `ensure_object` / `ensure_not_object` are aliases of `ensure_file` / `ensure_not_file`.

## 3. Attributes

The name parts follow `pathlib`: `basename` is `name`, `fname` is `stem`, `ext` is `suffix`. `dirname` is the name of the parent folder.

In [9]:
p = S3Path("my-bucket", "data", "2024", "report.tar.gz")
print(p.parts)
print(p.basename)
print(p.fname)
print(p.ext)
print(p.dirname)

['data', '2024', 'report.tar.gz']
report.tar.gz
report.tar
.gz
2024


`abspath` and `dirpath` are the Unix style paths from the bucket, which works like a drive; `root` is the bucket.

In [10]:
print(p.abspath)
print(p.dirpath)
print(p.root)

/data/2024/report.tar.gz
/data/2024/
S3Path('s3://my-bucket/')


`parent` is always a dir; `parents` goes up to the bucket (or to void for a relative path), like `pathlib.PurePath.parents`.

In [11]:
print(p.parent)
p.parents

S3Path('s3://my-bucket/data/2024/')


[S3Path('s3://my-bucket/data/2024/'),
 S3Path('s3://my-bucket/data/'),
 S3Path('s3://my-bucket/')]

In [12]:
S3Path.make_relpath("a/b/c.txt").parents

[S3RelPath('a/b/'), S3RelPath('a/'), S3VoidPath()]

A folder has no file name and no extension, so `fname` and `ext` raise. A folder's `basename` is the folder name.

In [13]:
d = S3Path("my-bucket", "data", "2024/")
print(d.basename)
show_error(lambda: d.ext)

2024
S3PathIsNotFileError: 's3://my-bucket/data/2024/' is not a file, it has no ext!


`arn` and `console_url` use the commercial `aws` partition; `get_arn` and `get_console_url` take another partition (`aws-cn`, `aws-us-gov`, `aws-eusc`) or a region.

In [14]:
p = S3Path("my-bucket", "folder", "file.txt")
print(p.arn)
print(p.get_arn("aws-us-gov"))
print(p.console_url)
print(p.get_console_url(region="us-west-2"))
print(p.get_console_url(partition="aws-cn"))

arn:aws:s3:::my-bucket/folder/file.txt
arn:aws-us-gov:s3:::my-bucket/folder/file.txt
https://console.aws.amazon.com/s3/object/my-bucket?prefix=folder/file.txt
https://us-west-2.console.aws.amazon.com/s3/object/my-bucket?region=us-west-2&prefix=folder/file.txt
https://console.amazonaws.cn/s3/object/my-bucket?prefix=folder/file.txt


## 4. Join

`/` and `joinpath` put parts **under a dir**. `joinpath` and a list on the right side of `/` join several parts at once.

In [15]:
bucket = S3Path("my-bucket")
print(bucket / "folder/" / "file.txt")
print(bucket.joinpath("folder", "file.txt"))
print(bucket / ["folder", "file.txt"])
print(bucket / "folder/sub/")

S3Path('s3://my-bucket/folder/file.txt')
S3Path('s3://my-bucket/folder/file.txt')
S3Path('s3://my-bucket/folder/file.txt')
S3Path('s3://my-bucket/folder/sub/')


A relative path can be joined too, which is how you rebuild a path under another folder. Joining the void path changes nothing.

In [16]:
rel = S3Path.make_relpath("2024/", "report.csv")
print(S3Path("my-bucket", "data/") / rel)
print(S3Path("my-bucket", "data/") / S3Path())

S3Path('s3://my-bucket/data/2024/report.csv')
S3Path('s3://my-bucket/data/')


**Joining onto a file is deprecated.** In `S3Path("my-bucket") / "folder" / "file.txt"`, the middle result `S3Path("my-bucket") / "folder"` is a file, because it has no trailing `/`. A file has no sub path, so this emits a `DeprecationWarning` today and will raise `S3PathIsNotFolderError` in 4.0.0. Write `"folder/"`, call `.to_dir()`, or join all parts at once as above.

In [17]:
import warnings

with warnings.catch_warnings(record=True) as records:
    warnings.simplefilter("always")
    p = S3Path("my-bucket") / "folder" / "file.txt"
print(p)
print(records[0].category.__name__)
print(records[0].message)

S3Path('s3://my-bucket/folder/file.txt')
DeprecationWarning
's3://my-bucket/folder' is a file. Joining onto a file is deprecated since 3.0.0, and will raise S3PathIsNotFolderError in 4.0.0. Join onto a dir instead: end the folder with '/' (e.g. ... / "folder/" / "file.txt"), call .to_dir() first, or join all parts at once (e.g. .joinpath("folder", "file.txt")).


## 5. Change a path

A path never changes; these methods return a new one. `change` replaces some attributes and keeps the others.

In [18]:
p = S3Path("my-bucket", "data", "2024", "report.csv")
print(p.change(new_bucket="other-bucket"))
print(p.change(new_ext=".json"))
print(p.change(new_fname="summary"))
print(p.change(new_basename="2024-12.csv"))
print(p.change(new_dirname="2025"))
print(p.change(new_dirpath="archive/2024/"))
print(p.change(new_bucket="other-bucket", new_abspath="new/key.csv"))

S3Path('s3://other-bucket/data/2024/report.csv')
S3Path('s3://my-bucket/data/2024/report.json')
S3Path('s3://my-bucket/data/2024/summary.csv')
S3Path('s3://my-bucket/data/2024/2024-12.csv')
S3Path('s3://my-bucket/data/2025/report.csv')
S3Path('s3://my-bucket/archive/2024/report.csv')
S3Path('s3://other-bucket/new/key.csv')


`new_basename` can turn a file into a folder (end it with `/`). Arguments that conflict or do not fit the path raise `ValueError`:

In [19]:
print(p.change(new_basename="report/"))
show_error(lambda: p.change(new_dirpath="a/", new_dirname="b"))
show_error(lambda: p.change(new_ext="json"))
show_error(lambda: S3Path("my-bucket", "data/").change(new_ext=".json"))

S3Path('s3://my-bucket/data/2024/report/')
ValueError: pass new_dirpath or new_dirname, not both
ValueError: new_ext has to be empty, or '.' followed by the extension, got 'json'
ValueError: new_fname and new_ext only work on a file, 's3://my-bucket/data/' is not a file


`to_dir` and `to_file` switch the type; `copy` returns an equal path (without the metadata loaded from S3, see [Metadata and Tags](../05-Metadata-and-Tags/index.ipynb)).

In [20]:
print(S3Path("my-bucket", "data").to_dir())
print(S3Path("my-bucket", "data/").to_file())
print(S3Path("my-bucket", "data/").copy() == S3Path("my-bucket", "data/"))
show_error(lambda: S3Path("my-bucket").to_file())

S3Path('s3://my-bucket/data/')
S3Path('s3://my-bucket/data')
True
S3PathIsNotFileError: 's3://my-bucket/' is a bucket root, it can not be converted to a file!


## 6. Relative paths

`relative_to` (or `-`) is the inverse of `/`: for a dir `base` and a path under it, `base / (p - base) == p`. This is how you mirror a folder somewhere else.

In [21]:
base = S3Path("my-bucket", "data/")
p = S3Path("my-bucket", "data", "2024", "report.csv")
rel = p.relative_to(base)
print(rel)
print(p - base)
print(base / rel == p)
print(S3Path("other-bucket", "backup/") / rel)

S3RelPath('2024/report.csv')
S3RelPath('2024/report.csv')
True
S3Path('s3://other-bucket/backup/2024/report.csv')


The relative path from a path to itself is void. When `p` is not under `base`, there is no relative path and it raises `ValueError`; that includes a file and a dir with the same name.

In [22]:
print(base - base)
show_error(lambda: p - S3Path("my-bucket", "logs/"))
show_error(lambda: p - S3Path("other-bucket", "data/"))
show_error(lambda: S3Path("my-bucket", "data/") - S3Path("my-bucket", "data"))

S3VoidPath()
ValueError: 's3://my-bucket/data/2024/report.csv' is not under 's3://my-bucket/logs/'
ValueError: 's3://my-bucket/data/2024/report.csv' is not under 's3://other-bucket/data/': they are in different buckets
ValueError: 's3://my-bucket/data/' is not under 's3://my-bucket/data': they have the same parts, but one is a file and the other is a dir, no relative path leads from one to the other


The base should be a dir. A file as the base (`S3Path("my-bucket", "data")`, without `/`) works for now with a `DeprecationWarning`, and raises `S3PathIsNotFolderError` in 4.0.0, for the same reason as joining onto a file. Relative paths can be relative to each other:

In [23]:
S3Path.make_relpath("a/b/c.txt") - S3Path.make_relpath("a/")

S3RelPath('b/c.txt')

`is_parent_of` tests the folder hierarchy; `is_prefix_of` tests the key as a string, the same as the `Prefix` of the S3 list APIs:

In [24]:
data = S3Path("my-bucket", "data/")
print(data.is_parent_of(S3Path("my-bucket", "data", "2024", "report.csv")))
print(data.is_parent_of(data))
print(S3Path("my-bucket", "data").is_prefix_of(S3Path("my-bucket", "data-2024.csv")))
print(data.is_prefix_of(S3Path("my-bucket", "data-2024.csv")))

True
False
True
False


## 7. Compare, sort and hash

Two paths are equal when they have the same bucket, the same parts and the same type, so a file and a folder of the same name differ. Paths are hashable and can be `set` members or `dict` keys.

In [25]:
print(S3Path("my-bucket", "a.txt") == S3Path("s3://my-bucket/a.txt"))
print(S3Path("my-bucket", "data") == S3Path("my-bucket", "data/"))
len({S3Path("my-bucket", "a.txt"), S3Path("s3://my-bucket/a.txt")})

True
False


1

`sorted` orders paths part by part, like `pathlib`: a folder comes right before its content. This is **not** the order of the S3 list APIs, which sort keys as strings (`-` comes before `/`); use `key=lambda p: p.key` for that order.

In [26]:
paths = [
    S3Path("my-bucket", "a-b.txt"),
    S3Path("my-bucket", "a", "b.txt"),
    S3Path("my-bucket", "a/"),
]
print(sorted(paths))
print(sorted(paths, key=lambda p: p.key))

[S3Path('s3://my-bucket/a/'), S3Path('s3://my-bucket/a/b.txt'), S3Path('s3://my-bucket/a-b.txt')]
[S3Path('s3://my-bucket/a-b.txt'), S3Path('s3://my-bucket/a/'), S3Path('s3://my-bucket/a/b.txt')]


## 8. Serialize

`to_dict` gives the three fields that define a path, which can be stored as JSON; `from_dict` builds the path back. Pickle works too (and keeps the metadata loaded from S3).

In [27]:
import json
import pickle

p = S3Path("my-bucket", "folder", "file.txt")
d = p.to_dict()
print(d)
print(S3Path.from_dict(json.loads(json.dumps(d))) == p)
print(pickle.loads(pickle.dumps(p)) == p)

{'bucket': 'my-bucket', 'parts': ['folder', 'file.txt'], 'is_dir': False}
True
True


## Common pitfalls

- **Forgetting the trailing `/`.** `S3Path("my-bucket", "folder")` is a file. Folder methods such as `iter_objects` raise `S3PathIsNotFolderError` on it, and joining onto it is deprecated.
- **Sorting and S3 order.** `sorted(paths)` is not the order S3 returns; see section 7.
- **Attributes on a path.** `S3Path` has `__slots__`, so `p.my_attr = 1` raises `AttributeError`. Keep your own data in a `dict` keyed by the path.

## See also

- [Quick Start](../01-Quick-Start/index.ipynb)
- [Read and Write](../03-Read-and-Write/index.ipynb): the first chapter that calls AWS.